# Data Viz Assignment

### Importing packages

In [13]:
# importing required packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tabulate
import pycountry as pc
import pycountry_convert as pcc
import prophet as prt
import statsmodels as sm
import datetime as dt
import streamlit as st
import plotly.express as px

### Loading datasets

In [14]:
# loading temperature anomaly dataset
temp_df = pd.read_csv("../data/annual-temperature-anomalies.csv")

# loading co2 emissions dataset
co2_df = pd.read_csv("../data/owid-co2-data.csv")

### Merging data

In [15]:
# merging temperature and co2 datasets
# merged on year and country
df = pd.merge(temp_df, co2_df,
              left_on = ["Entity", "Year"],
              right_on = ["country", "year"],
              how = "inner")

### Dataset overview

Initial exploration of the datasets to assess dataset structure and contents.

In [16]:
# creating a function to assess data structure and contents
# using a report like structure for readability
def info_stats(df):

    # creating title
    print("\n" + "="*60)
    print("DATASET OVERVIEW")  
    print("="*60)

    # getting column names
    print("\n Columns:")
    print(list(df.columns.values)) 
    
    # getting row and columns totals
    print("\n Shape (rows, columns):")
    print(df.shape) 
    
    # viewing top 5 rows
    print("\n First 5 rows:")
    print(df.head()) 
    
    # viewing last 5 rows
    print("\n Last 5 rows:")
    print(df.tail()) 
    
    # collecting data info
    print("\n Data Info:")
    df.info()
    
    # getting summary statistics
    print("\n Summary Statistics:")
    summary_statistics = df.describe().round(2)
    table = tabulate.tabulate(summary_statistics, headers='keys', tablefmt='pretty')
    print(table)
    
    # adding a seperator for readabilty
    print("\n" + "="*60 + "\n")

In [17]:
# applying the info_stats function created in the previous code block
info_stats(df)


DATASET OVERVIEW

 Columns:
['Entity', 'Code', 'Year', 'Temperature anomaly', 'country', 'year', 'iso_code', 'population', 'gdp', 'cement_co2', 'cement_co2_per_capita', 'co2', 'co2_growth_abs', 'co2_growth_prct', 'co2_including_luc', 'co2_including_luc_growth_abs', 'co2_including_luc_growth_prct', 'co2_including_luc_per_capita', 'co2_including_luc_per_gdp', 'co2_including_luc_per_unit_energy', 'co2_per_capita', 'co2_per_gdp', 'co2_per_unit_energy', 'coal_co2', 'coal_co2_per_capita', 'consumption_co2', 'consumption_co2_per_capita', 'consumption_co2_per_gdp', 'cumulative_cement_co2', 'cumulative_co2', 'cumulative_co2_including_luc', 'cumulative_coal_co2', 'cumulative_flaring_co2', 'cumulative_gas_co2', 'cumulative_luc_co2', 'cumulative_oil_co2', 'cumulative_other_co2', 'energy_per_capita', 'energy_per_gdp', 'flaring_co2', 'flaring_co2_per_capita', 'gas_co2', 'gas_co2_per_capita', 'ghg_excluding_lucf_per_capita', 'ghg_per_capita', 'land_use_change_co2', 'land_use_change_co2_per_capita', 

### Data cleaning and preprocessing

Cleaning the data by removing unwanted columns and rows. This includes rows that contain NA values in key statistical columns needed for the analysis.

In [18]:
# defining column list for filtering
columns = ["Year", "country", "iso_code", "population", "gdp", "energy_per_capita", "Temperature anomaly",
           "co2", "co2_per_capita", "co2_including_luc",
           "land_use_change_co2", "coal_co2", "oil_co2", "gas_co2", "share_global_co2", "temperature_change_from_co2"]

In [19]:
# filtering dataframe for only selected columns
df = df[columns]
df.shape

(15895, 16)

In [20]:
world_df = df[df['country'] == "World"]
world_df

,Year,country,iso_code,population,gdp,energy_per_capita,Temperature anomaly,co2,co2_per_capita,co2_including_luc,land_use_change_co2,coal_co2,oil_co2,gas_co2,share_global_co2,temperature_change_from_co2
15555,1940,World,NaN,2.291980e+09,7.646890e+12,NaN,-0.686135,4884.661621,2.131198,10628.067383,5743.405762,3795.503174,900.980530,153.519745,100.0,0.247305
15556,1941,World,NaN,2.312099e+09,NaN,NaN,-0.641774,4999.090332,2.162144,10890.240234,5891.149902,3885.962891,912.336914,153.463226,100.0,0.252204
15557,1942,World,NaN,2.331135e+09,NaN,NaN,-0.732430,4982.523438,2.137381,10887.828125,5905.305176,3910.556152,863.507507,166.048950,100.0,0.257101
15558,1943,World,NaN,2.349015e+09,NaN,NaN,-0.726379,5075.272461,2.160596,10969.573242,5894.301270,3926.686035,926.590759,182.899902,100.0,0.262035
15559,1944,World,NaN,2.366919e+09,NaN,NaN,-0.534997,5162.485840,2.181100,10988.550781,5826.065430,3869.107178,1068.786133,197.600830,100.0,0.266977
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15635,2020,World,NaN,7.887001e+09,1.185900e+14,20032.187500,0.432163,35158.230469,4.457744,39674.769531,4516.539551,14267.035156,10932.341797,7520.819824,100.0,1.141716
15636,2021,World,NaN,7.954448e+09,1.260048e+14,20874.294922,0.273691,36866.863281,4.634748,41523.964844,4657.103027,14969.249023,11513.166016,7878.698242,100.0,1.160024
15637,2022,World,NaN,8.021407e+09,1.301126e+14,21076.292969,0.300865,37527.773438,4.678452,42313.164062,4785.391602,15449.176758,11914.912109,7758.745605,100.0,1.178648
15638,2023,World,NaN,8.091735e+09,NaN,21285.767578,0.600733,38094.039062,4.707772,42839.593750,4745.551270,15636.373047,12278.241211,7802.845215,100.0,1.197455


In [21]:
world_df = world_df[(world_df["Year"] <= 2022) & (world_df["Year"] >= 1972)]
world_df

,Year,country,iso_code,population,gdp,energy_per_capita,Temperature anomaly,co2,co2_per_capita,co2_including_luc,land_use_change_co2,coal_co2,oil_co2,gas_co2,share_global_co2,temperature_change_from_co2
15587,1972,World,NaN,3.844918e+09,NaN,16308.279297,-0.595112,16224.579102,4.219747,22662.667969,6438.087891,5691.283203,7764.705078,2052.348877,100.000000,0.466356
15588,1973,World,NaN,3.920805e+09,NaN,16919.394531,-0.491711,17077.285156,4.355556,23331.548828,6254.264648,5860.898438,8219.690430,2202.722656,100.000000,0.476618
15589,1974,World,NaN,3.996416e+09,NaN,16689.080078,-0.767671,16997.912109,4.253289,23310.912109,6312.999023,5857.693359,8099.290039,2256.540527,100.000000,0.486888
15590,1975,World,NaN,4.070735e+09,NaN,16497.583984,-0.742727,16988.476562,4.173319,23111.972656,6123.496582,6018.314941,8005.122070,2241.266357,100.000000,0.497088
15591,1976,World,NaN,4.144246e+09,NaN,17070.597656,-0.811458,17862.492188,4.310191,23922.773438,6060.280762,6226.339844,8475.641602,2359.009277,100.000000,0.507656
15592,1977,World,NaN,4.217864e+09,NaN,17379.927734,-0.525560,18358.039062,4.352449,24978.128906,6620.090820,6375.023438,8767.405273,2411.720459,100.000000,0.518694
15593,1978,World,NaN,4.292097e+09,NaN,17748.673828,-0.584493,19024.742188,4.432505,25588.640625,6563.897461,6510.127930,9133.722656,2541.737305,100.000000,0.530005
15594,1979,World,NaN,4.368540e+09,NaN,18014.863281,-0.429016,19508.589844,4.465701,25364.417969,5855.829102,6799.881836,9224.317383,2668.064941,100.000000,0.541213
15595,1980,World,NaN,4.447606e+09,3.198262e+13,17539.898438,-0.297785,19409.792969,4.364099,25411.107422,6001.314453,6984.989746,8912.473633,2741.514160,100.000000,0.552448
15596,1981,World,NaN,4.528777e+09,NaN,17146.964844,-0.264391,18879.755859,4.168842,24716.337891,5836.581055,6942.941406,8483.404297,2763.373047,100.000000,0.563384


In [22]:
# Getting rid of any rows containing an NA
df = df.dropna()
df.shape

(4448, 16)

Creating a list of countries with the correct country name added for use of ISO codes

In [23]:
# Manual lookup for countries that dont have correct ISO name
country_corrections = {
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Democratic Republic of Congo": "Congo, The Democratic Republic of the",
    "Congo": "Congo",
    "Tanzania": "Tanzania, United Republic of",
    "South Korea": "Korea, Republic of",
    "North Korea": "Korea, Democratic People's Republic of",
    "Laos": "Lao People's Democratic Republic",
    "Syria": "Syrian Arab Republic",
    "Russia": "Russian Federation",
    "Vietnam": "Viet Nam",
    "Bolivia": "Bolivia, Plurinational State of",
    "Moldova": "Moldova, Republic of",
    "Venezuela": "Venezuela, Bolivarian Republic of",
    "Iran": "Iran, Islamic Republic of",
    "Palestine": "Palestine, State of",
    "Brunei": "Brunei Darussalam",
    "Cape Verde": "Cabo Verde",
    "Swaziland": "Eswatini",
    "Turkey": "Türkiye"
}

In [24]:
# removing world
country_df = df[df['country'] != "World"]
country_df

,Year,country,iso_code,population,gdp,energy_per_capita,Temperature anomaly,co2,co2_per_capita,co2_including_luc,land_use_change_co2,coal_co2,oil_co2,gas_co2,share_global_co2,temperature_change_from_co2
40,1980,Afghanistan,AFG,13169313.0,1.532984e+10,481.208618,-0.059716,1.756302,0.133363,4.339825,2.583523,0.315762,0.925256,0.187254,0.009049,0.000467
41,1981,Afghanistan,AFG,11937586.0,1.564534e+10,610.638977,-0.327458,1.978463,0.165734,4.677109,2.698646,0.333424,1.014928,0.304112,0.010479,0.000470
42,1982,Afghanistan,AFG,10991380.0,1.598041e+10,717.766418,-1.718081,2.094581,0.190566,4.267553,2.172972,0.384720,0.992944,0.395712,0.011183,0.000472
43,1983,Afghanistan,AFG,10917985.0,1.675533e+10,905.126648,-0.780836,2.519954,0.230808,4.644854,2.124900,0.384720,1.220112,0.615552,0.013332,0.000475
44,1984,Afghanistan,AFG,11190222.0,1.707215e+10,887.370972,-0.886393,2.821540,0.252143,4.676733,1.855193,0.392556,1.133644,0.931863,0.014501,0.000478
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15718,2018,Yemen,YEM,34085181.0,6.527121e+10,1109.117920,-0.031227,11.503876,0.337504,11.394616,-0.109260,0.271136,9.504416,0.040304,0.031317,0.000283
15719,2019,Yemen,YEM,35111416.0,6.618501e+10,1085.563599,0.521249,11.787783,0.335725,11.630597,-0.157186,0.278378,9.358638,0.040292,0.031785,0.000288
15720,2020,Yemen,YEM,36134867.0,6.055929e+10,918.665466,0.108743,10.757014,0.297691,10.621593,-0.135421,0.252816,8.137744,0.040304,0.030596,0.000293
15721,2021,Yemen,YEM,37140234.0,5.995369e+10,863.311096,0.068503,10.607956,0.285619,10.543360,-0.064596,0.252730,7.819967,0.040290,0.028774,0.000298


In [25]:
# adding continent column for continental analysis

def continent_name(country_name):
  # correcting names for usable ISO name
  country_name = country_corrections.get(country_name, country_name)

  # lookup name and find matching continent code and name
  country = pc.countries.lookup(country_name)
  alpha2 = country.alpha_2
  continent_code = pcc.country_alpha2_to_continent_code(alpha2)
  continent_name = pcc.convert_continent_code_to_continent_name(continent_code)

  return continent_name

# Applying function to add continent name
country_df['Continent'] = country_df['country'].apply(continent_name)
country_df

,Year,country,iso_code,population,gdp,energy_per_capita,Temperature anomaly,co2,co2_per_capita,co2_including_luc,land_use_change_co2,coal_co2,oil_co2,gas_co2,share_global_co2,temperature_change_from_co2,Continent
40,1980,Afghanistan,AFG,13169313.0,1.532984e+10,481.208618,-0.059716,1.756302,0.133363,4.339825,2.583523,0.315762,0.925256,0.187254,0.009049,0.000467,Asia
41,1981,Afghanistan,AFG,11937586.0,1.564534e+10,610.638977,-0.327458,1.978463,0.165734,4.677109,2.698646,0.333424,1.014928,0.304112,0.010479,0.000470,Asia
42,1982,Afghanistan,AFG,10991380.0,1.598041e+10,717.766418,-1.718081,2.094581,0.190566,4.267553,2.172972,0.384720,0.992944,0.395712,0.011183,0.000472,Asia
43,1983,Afghanistan,AFG,10917985.0,1.675533e+10,905.126648,-0.780836,2.519954,0.230808,4.644854,2.124900,0.384720,1.220112,0.615552,0.013332,0.000475,Asia
44,1984,Afghanistan,AFG,11190222.0,1.707215e+10,887.370972,-0.886393,2.821540,0.252143,4.676733,1.855193,0.392556,1.133644,0.931863,0.014501,0.000478,Asia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15718,2018,Yemen,YEM,34085181.0,6.527121e+10,1109.117920,-0.031227,11.503876,0.337504,11.394616,-0.109260,0.271136,9.504416,0.040304,0.031317,0.000283,Asia
15719,2019,Yemen,YEM,35111416.0,6.618501e+10,1085.563599,0.521249,11.787783,0.335725,11.630597,-0.157186,0.278378,9.358638,0.040292,0.031785,0.000288,Asia
15720,2020,Yemen,YEM,36134867.0,6.055929e+10,918.665466,0.108743,10.757014,0.297691,10.621593,-0.135421,0.252816,8.137744,0.040304,0.030596,0.000293,Asia
15721,2021,Yemen,YEM,37140234.0,5.995369e+10,863.311096,0.068503,10.607956,0.285619,10.543360,-0.064596,0.252730,7.819967,0.040290,0.028774,0.000298,Asia


In [ ]:
country_df = country_df[(country_df["Year"] <= 2022) & (country_df["Year"] >= 1972)]

,Year,country,iso_code,population,gdp,energy_per_capita,Temperature anomaly,co2,co2_per_capita,co2_including_luc,land_use_change_co2,coal_co2,oil_co2,gas_co2,share_global_co2,temperature_change_from_co2,Continent
40,1980,Afghanistan,AFG,13169313.0,1.532984e+10,481.208618,-0.059716,1.756302,0.133363,4.339825,2.583523,0.315762,0.925256,0.187254,0.009049,0.000467,Asia
41,1981,Afghanistan,AFG,11937586.0,1.564534e+10,610.638977,-0.327458,1.978463,0.165734,4.677109,2.698646,0.333424,1.014928,0.304112,0.010479,0.000470,Asia
42,1982,Afghanistan,AFG,10991380.0,1.598041e+10,717.766418,-1.718081,2.094581,0.190566,4.267553,2.172972,0.384720,0.992944,0.395712,0.011183,0.000472,Asia
43,1983,Afghanistan,AFG,10917985.0,1.675533e+10,905.126648,-0.780836,2.519954,0.230808,4.644854,2.124900,0.384720,1.220112,0.615552,0.013332,0.000475,Asia
44,1984,Afghanistan,AFG,11190222.0,1.707215e+10,887.370972,-0.886393,2.821540,0.252143,4.676733,1.855193,0.392556,1.133644,0.931863,0.014501,0.000478,Asia
45,1985,Afghanistan,AFG,11426855.0,1.710848e+10,842.730957,-0.298625,3.501422,0.306420,5.075586,1.574164,0.399794,1.547825,1.192046,0.017381,0.000481,Asia
46,1986,Afghanistan,AFG,11420074.0,1.764134e+10,875.000305,-1.225358,3.133645,0.274398,4.406152,1.272507,0.425024,1.139504,1.201792,0.015332,0.000484,Asia
47,1987,Afghanistan,AFG,11387822.0,1.581082e+10,1486.686646,0.099336,3.113826,0.273435,4.137804,1.023978,0.442824,2.012838,0.391588,0.014753,0.000486,Asia
48,1988,Afghanistan,AFG,11523299.0,1.449907e+10,2697.882080,0.384678,2.856896,0.247923,3.866218,1.009322,0.366400,1.821008,0.439680,0.013034,0.000489,Asia
49,1989,Afghanistan,AFG,11874089.0,1.348950e+10,2569.341797,-1.639677,2.764855,0.232848,3.688476,0.923621,0.337088,1.864976,0.479984,0.012454,0.000491,Asia


In [31]:
world_df.to_csv("../Data/world_data.csv", index=False)
country_df.to_csv("../Data/country_data.csv", index=False)